### **회귀 분석**
#### **상관 계수**
- 두 변수 간의 선형적인 관계가 어느 정도 강한지를 나타냄
- -1 <= r <= 1
     - 1에 가까울수록 **강한 양의 선형 관계**
     - -1에 가까울수록 **강한 음의 선형 관계**
- `pd.DataFrame.corr(method='pearson', numeric_only=True)`
     - method (상관 관계 방법)
        - pearson 피어슨 상관계수 (default)
        - kendall 켄달 상관계수
        - spearman 스피어만 상관계수

In [1]:
# 데이터프레임 정의
import pandas as pd

data = {
    '키': [150, 160, 170, 175, 165],
    '몸무게': [42, 50, 70, 64, 56]
}

df = pd.DataFrame(data)
df

,키,몸무게
0,150,42
1,160,50
2,170,70
3,175,64
4,165,56


In [2]:
df.corr()

,키,몸무게
키,1.000000,0.919509
몸무게,0.919509,1.000000


In [3]:
print(df.corr().iloc[0,1])
print(df['키'].corr(df['몸무게']))

0.9195090879163764
0.9195090879163765


In [4]:
## 다양한 상관 계수 적용
# pearson (기본값)
print(df.corr())

# kendall
print(df.corr(method='kendall'))

# spearman
print(df.corr(method='spearman'))

print('\n====== 상관계수와 p-value ======')
# 피어슨 상관계수와 p-value 계산
from scipy import stats
print(stats.pearsonr(df['몸무게'], df['키']))

# 켄달 상관계수와 p-value 계산
print(stats.kendalltau(df['몸무게'], df['키']))

# 스피어만 상관계수와 p-value 계산
print(stats.spearmanr(df['몸무게'], df['키']))

            키       몸무게
키    1.000000  0.919509
몸무게  0.919509  1.000000
       키  몸무게
키    1.0  0.8
몸무게  0.8  1.0
       키  몸무게
키    1.0  0.9
몸무게  0.9  1.0

====== 상관계수와 p-value ======
PearsonRResult(statistic=0.9195090879163764, pvalue=0.027079456895589476)
SignificanceResult(statistic=0.7999999999999999, pvalue=0.08333333333333333)
SignificanceResult(statistic=0.8999999999999998, pvalue=0.03738607346849874)


---
#### **단순 선형 회귀 분석**
회귀 분석의 유형은 독립변수의 개수에 따라 달라짐.
- 단순 회귀 분석 : 독립변수가 1개
- 다중 회귀 분석 : 독립변수가 2개 이상

**단순 선형 회귀식**


$$
y_i = \alpha + \beta x_i + \varepsilon_i
$$

- $y_i$ : 종속 변수 (실제값)  
- $x_i$ : 독립 변수 (입력값)  
- $\alpha$ : 절편 (Intercept)  
- $\beta$ : 기울기 (Slope)  
- $\varepsilon_i$ : 오차 항 (Error term)


**Q. 다음은 20명의 키와 몸무게에 관한 정보다. 이 데이터를 바탕으로 회귀 모델을 구축하고 각 소문제의 값을 구하시오.**

In [5]:
import pandas as pd

data = {
    '키': [150, 160, 170, 175, 165, 155, 172, 168, 174, 158,
          162, 173, 156, 159, 167, 163, 171, 169, 176, 161],
    '몸무게': [42, 50, 70, 64, 56, 48, 68, 60, 65, 52,
            54, 67, 49, 51, 58, 55, 69, 61, 66, 53]
}
df = pd.DataFrame(data)
df.head()

,키,몸무게
0,150,42
1,160,50
2,170,70
3,175,64
4,165,56


1. 주어진 데이터로 최소제곱법을 이용한 단순 선형 회귀 모델을 구축하고 통계적 요약을 출력하시오.

In [6]:
from statsmodels.formula.api import ols

model = ols('키 ~ 몸무게', data=df).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                      키   R-squared:                       0.892
Model:                            OLS   Adj. R-squared:                  0.886
Method:                 Least Squares   F-statistic:                     148.0
Date:                Mon, 16 Jun 2025   Prob (F-statistic):           4.04e-10
Time:                        19:54:31   Log-Likelihood:                -45.761
No. Observations:                  20   AIC:                             95.52
Df Residuals:                      18   BIC:                             97.51
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    115.0676      4.158     27.671      0.0

2. 회귀 모델의 결정 계수를 구하시오.

In [7]:
print(model.rsquared)

0.8915914350087263


3. 회귀 모델에서 회귀 계수(기울기와 절편)을 구하시오.

In [8]:
print(model.params['몸무게'])
print(model.params['Intercept'])

0.8658438852380215
115.06763904471848


4. 회귀 모델에서 몸무게의 회귀 계수가 통계적으로 유의한지 검정했을 때의 p-value를 구하시오.

In [9]:
print('{:.10f}'.format(model.pvalues['몸무게']))

0.0000000004


5. 회귀 모델을 사용해 몸무게가 67일 때의 예측 키를 구하시오.

In [10]:
new_data = pd.DataFrame({'몸무게':[67]})
result = model.predict(new_data)
print(result[0])

173.07917935566593


6. 회귀 모델의 잔차 제곱합을 구하시오.

In [11]:
df['diff'] = df['키'] - model.predict(df)
print(sum(df['diff']**2))

113.74226638884433


회귀 모델의 MSE를 구하시오.

In [12]:
df['diff'] = df['키'] - model.predict(df)
MSE = (df['diff'] ** 2).mean() ## 평균 제곱 오차 -> 잔차의 평균
print(MSE)

5.687113319442217


In [13]:
from sklearn.metrics import mean_squared_error

pred = model.predict(df['몸무게'])
MSE = mean_squared_error(df['키'], pred)
print(MSE)

5.687113319442217


**Q. 몸무게의 95% 신뢰 구간을 구하시오.**

In [14]:
from scipy import stats
from statsmodels.formula.api import ols

model = ols('키 ~ 몸무게', data=df).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                      키   R-squared:                       0.892
Model:                            OLS   Adj. R-squared:                  0.886
Method:                 Least Squares   F-statistic:                     148.0
Date:                Mon, 16 Jun 2025   Prob (F-statistic):           4.04e-10
Time:                        19:54:32   Log-Likelihood:                -45.761
No. Observations:                  20   AIC:                             95.52
Df Residuals:                      18   BIC:                             97.51
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    115.0676      4.158     27.671      0.0

In [15]:
print(model.conf_int(alpha=0.05).loc['몸무게'])
## 만약 90% 의 신뢰 구간을 구하려면 alpha를 0.1로 변경

0    0.716337
1    1.015351
Name: 몸무게, dtype: float64


**Q. 몸무게가 50일 때 예측 키의 신뢰 구간과 예측 구간을 구하시오.**

In [16]:
new_data = pd.DataFrame({'몸무게':[50]})

pred = model.get_prediction(new_data)
result = pred.summary_frame(alpha=0.05)
result
## mean_ci - 신뢰 구간
## obs_ci - 예측 구간

,mean,mean_se,mean_ci_lower,mean_ci_upper,obs_ci_lower,obs_ci_upper
0,158.359833,0.794986,156.68963,160.030037,152.820798,163.898869


---
#### **다중 선형 회귀 분석**

$$
Y = \alpha + \beta_1 X_1 + \beta_2 X_2 + \cdots + \beta_p X_p + \varepsilon
$$

Q. 다음은 매출액, 광고비, 직원 수에 관한 데이터다. 광고비와 직원 수는 독립변수고, 매출액은 종속변수다. 다중 선형 회귀 모델을 구축하고 각 소문제의 값을 구하시오.

In [17]:
import pandas as pd
data = {
    '매출액': [300, 320, 250, 360, 315, 328, 310, 335, 326, 280,
            290, 300, 315, 328, 310, 335, 300, 400, 500, 600],
    '광고비': [70, 75, 30, 80, 72, 77, 70, 82, 70, 80,
            68, 90, 72, 77, 70, 82, 40, 20, 75, 80],
    '직원수': [15, 16, 14, 20, 19, 17, 16, 19, 15, 20,
            14, 5, 16, 17, 16, 14, 30, 40, 10, 50]
    }
df = pd.DataFrame(data)
df.head(3)

,매출액,광고비,직원수
0,300,70,15
1,320,75,16
2,250,30,14


1. 주어진 데이터로 최소제곱법을 이용한 다중 선형 회귀 모델을 구축하고 통계적 요약을 출력하시오.

In [18]:
from statsmodels.formula.api import ols

model = ols('매출액 ~ 광고비 + 직원수', data=df).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                    매출액   R-squared:                       0.512
Model:                            OLS   Adj. R-squared:                  0.454
Method:                 Least Squares   F-statistic:                     8.907
Date:                Mon, 16 Jun 2025   Prob (F-statistic):            0.00226
Time:                        19:54:32   Log-Likelihood:                -108.22
No. Observations:                  20   AIC:                             222.4
Df Residuals:                      17   BIC:                             225.4
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    101.0239     71.716      1.409      0.1

2. 광고비와 매출액의 상관 계수를 구하시오.

In [19]:
df['광고비'].corr(df['매출액'])

0.13316981737040343

3. 광고비와 매출액의 t-검정의 p-value를 구하시오.

In [20]:
from scipy import stats

stats.pearsonr(df['광고비'], df['매출액'])

PearsonRResult(statistic=0.13316981737040345, pvalue=0.5756778801904272)

4. 회귀 모델의 결정 계수를 구하시오.

In [22]:
print(model.rsquared)

0.5116964327009041


5. 회귀 모델에서 회귀 계수(기울기와 절편)를 구하시오.

In [24]:
print(model.params)

Intercept    101.023872
광고비            1.819427
직원수            5.928756
dtype: float64


6. 회귀 모델에서 광고비의 회귀 계수가 통계적으로 유의한지 검정했을 때의 p-value를 구하시오.

In [25]:
model.pvalues['광고비']

0.0376435064769604

7. 광고비 50, 직원 수 20인 데이터가 있을 때 구축한 회귀 모델에서의 예상 매출액을 구하시오.

In [26]:
new_data = pd.DataFrame({'광고비':[50],'직원수':[20]})
result = model.predict(new_data)
result

0    310.57033
dtype: float64

8. 회귀 모델의 잔차의 제곱합을 구하시오.

In [28]:
df['diff'] = df['매출액'] - model.predict(df)
print(sum(df['diff'] ** 2))

# 잔차를 구하는 다른 방법
print(sum(model.resid**2))

58686.178271561075
58686.178271561075


9. 회귀 모델의 MSE를 구하시오.

In [30]:
print((df['diff'] ** 2).mean())
print((model.resid**2).mean())

2934.3089135780538
2934.3089135780538


10. 각 변수별 95%의 신뢰 구간을 구하시오.

In [33]:
model.conf_int(alpha=0.05)

,0,1
Intercept,-50.283684,252.331429
광고비,0.116785,3.522069
직원수,2.912406,8.945105


11. 광고비 45, 직원수 22일 때, 95%의 신뢰 구간과 예측 구간을 구하시오.

In [36]:
new_data = pd.DataFrame({'광고비':[45],'직원수':[22]})
pred = model.get_prediction(new_data)
result = pred.summary_frame(alpha=0.05)
result

,mean,mean_se,mean_ci_lower,mean_ci_upper,obs_ci_lower,obs_ci_upper
0,313.330707,22.502058,265.855514,360.8059,180.58875,446.072663


---
#### **범주형 변수**
##### **범주형 변수 자동 원-핫 인코딩**

In [39]:
import pandas as pd
from statsmodels.formula.api import ols

df = pd.read_csv('study.csv')
model = ols('score ~ study_hours + material_type', data=df).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                  score   R-squared:                       0.969
Model:                            OLS   Adj. R-squared:                  0.968
Method:                 Least Squares   F-statistic:                     991.9
Date:                Mon, 16 Jun 2025   Prob (F-statistic):           4.42e-72
Time:                        20:20:25   Log-Likelihood:                -238.89
No. Observations:                 100   AIC:                             485.8
Df Residuals:                      96   BIC:                             496.2
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
Intercept              59.2111    